In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from skimage import io  #read images
from skimage.transform import rescale, resize, downscale_local_mean  #there are other scaling methods
from skimage.filters import roberts, sobel, scharr, prewitt, gabor_kernel

import cv2
import shutil

import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgbClassifier

from sklearn.metrics import precision_recall_curve, roc_curve, auc
from sklearn.metrics import classification_report
from sklearn.model_selection import learning_curve, GridSearchCV

from sklearn.decomposition import PCA
import Augmentor

In [ ]:
trainingData = pd.read_csv("trainset.csv") #, index_col=0)
testingData = pd.read_csv("testset.csv") #, index_col=0)

In [ ]:
trainingData["Class"].value_counts()

In [ ]:
#Scaling Images' size
img = cv2.imread("images/images/1006.png") #, as_gray = True)
resized_img = cv2.resize(img, (500, 500))
gray_img = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
imgee = resized_img.reshape(1, -1)
#imgee.shape
img_features = imgee.flatten()
img_features.shape

In [ ]:
counts = trainingData['Class'].value_counts()
plt.pie(counts.tolist(), labels = trainingData['Class'].unique(), autopct='%.0f%%')
plt.title("Target Distribution")
plt.legend()
plt.show()

In [ ]:
trainingData.head()

In [ ]:
count = 0
for image in os.listdir("images"):
  image_id = image.split('.')[0]
  count = count +1
  print(type(image_id))
  if count == 1:
    break

In [ ]:
img = cv2.imread("images/images/1006.png") #, as_gray = True)
gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray_img.shape

# getGaborKernel

In [ ]:
resized_img = cv2.resize(gray_img, (200, 200))

ksize = 5
sigma = 3
theta = 1 * np.pi/4
lamda = 1 * np.pi/4  #Wavelength
gamma = 0.5
phi = 0 #phase offset
kernel = cv2.getGaborKernel((ksize, ksize), sigma, theta, lamda, gamma, phi, ktype = cv2.CV_32F)

getGabor_filtered_img = cv2.filter2D(resized_img, cv2.CV_8U, kernel)


# Display the resized and filtered images using matplotlib
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(resized_img, cmap='gray')
plt.title("Resized Image")

plt.subplot(1, 2, 2)
plt.imshow(getGabor_filtered_img, cmap='gray')
plt.title("Filtered Image")

plt.show()

# bilateralFilter()

In [ ]:
# Assuming you have loaded or obtained the 'gray_img' properly
resized_img = cv2.resize(gray_img, (200, 200))

# Apply bilateral filter
d = 9         # Diameter of pixel neighborhood
sigmaColor = 75  # Filter sigma in the color space
sigmaSpace = 75  # Filter sigma in the coordinate space
bilateralFilter_filtered_img = cv2.bilateralFilter(resized_img, d, sigmaColor, sigmaSpace)

# Display the resized and filtered images using matplotlib
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(resized_img, cmap='gray')
plt.title("Resized Image")

plt.subplot(1, 2, 2)
plt.imshow(bilateralFilter_filtered_img, cmap='gray')
plt.title("Filtered Image")

plt.show()


In [ ]:
bilateralFilter_filtered_img.shape

# GaussianBlur()

In [ ]:
# Assuming you have loaded or obtained the 'gray_img' properly
resized_img = cv2.resize(gray_img, (200, 200))

ksize = 5
sigma = 3

# Create a Gaussian blur kernel using cv2.GaussianBlur
kernel = cv2.GaussianBlur(resized_img, (ksize, ksize), sigma)

# Apply the Gaussian blur using filter2D (not necessary for Gaussian blur, but shown for consistency)
GaussianBlur_filtered_img = cv2.filter2D(resized_img, cv2.CV_8U, kernel)

# Display the resized and filtered images using matplotlib
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(resized_img, cmap='gray')
plt.title("Resized Image")

plt.subplot(1, 2, 2)
plt.imshow(GaussianBlur_filtered_img, cmap='gray')
plt.title("Filtered Image")

plt.show()

# Sobel()

In [ ]:
# Assuming you have loaded or obtained the 'gray_img' properly
resized_img = cv2.resize(gray_img, (200, 200))

# Apply the Sobel filter for edge detection
sobel_x = cv2.Sobel(resized_img, cv2.CV_64F, 1, 0, ksize=5)
sobel_y = cv2.Sobel(resized_img, cv2.CV_64F, 0, 1, ksize=5)

# Calculate the gradient magnitude
gradient_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)

# Convert the gradient magnitude to 8-bit unsigned integer
gradient_magnitude = cv2.convertScaleAbs(gradient_magnitude)

# Display the resized image and gradient magnitude using matplotlib
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(resized_img, cmap='gray')
plt.title("Resized Image")

plt.subplot(1, 2, 2)
plt.imshow(gradient_magnitude, cmap='gray')
plt.title("Gradient Magnitude (Sobel)")

plt.show()


# Laplacian()

In [ ]:
# Assuming you have loaded or obtained the 'gray_img' properly
resized_img = cv2.resize(gray_img, (200, 200))

# Apply the Laplacian filter
Laplacian_filtered_img = cv2.Laplacian(resized_img, cv2.CV_8U)

# Display the resized and filtered images using matplotlib
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(resized_img, cmap='gray')
plt.title("Resized Image")

plt.subplot(1, 2, 2)
plt.imshow(Laplacian_filtered_img, cmap='gray')
plt.title("Filtered Image")

plt.show()


# medianBlur()


In [ ]:
# Assuming you have loaded or obtained the 'gray_img' properly
resized_img = cv2.resize(gray_img, (200, 200))

ksize = 5

# Apply median blur using cv2.medianBlur
medianBlur_filtered_img = cv2.medianBlur(resized_img, ksize)

# Display the resized and filtered images using matplotlib
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(resized_img, cmap='gray')
plt.title("Resized Image")

plt.subplot(1, 2, 2)
plt.imshow(medianBlur_filtered_img, cmap='gray')
plt.title("Filtered Image (Median Blur)")

plt.show()


# morphologyEx()

In [ ]:
# Assuming you have loaded or obtained the 'gray_img' properly
resized_img = cv2.resize(gray_img, (200, 200))

# Define the kernel for morphological operation
kernel_morphologyEx = np.ones((ksize, ksize), np.uint8)

# Apply dilation using cv2.morphologyEx()
morphologyEx_dilated_img = cv2.morphologyEx(resized_img, cv2.MORPH_DILATE, kernel_morphologyEx)

# Display the resized and dilated images using matplotlib
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(resized_img, cmap='gray')
plt.title("Resized Image")

plt.subplot(1, 2, 2)
plt.imshow(morphologyEx_dilated_img, cmap='gray')
plt.title("Dilated Image")

plt.show()


# pyrMeanShiftFiltering()

In [ ]:
# Assuming you have loaded or obtained the 'gray_img' properly
resized_img = cv2.resize(gray_img, (200, 200))

# Convert the grayscale image to a 3-channel image (color image)
resized_color_img = cv2.cvtColor(resized_img, cv2.COLOR_GRAY2BGR)

# Parameters for pyrMeanShiftFiltering
spatial_radius = 15
color_radius = 25

# Apply pyrMeanShiftFiltering
filtered_color_img = cv2.pyrMeanShiftFiltering(resized_color_img, spatial_radius, color_radius)

# Convert the filtered color image back to grayscale
pyrMeanShiftFiltering_filtered_img = cv2.cvtColor(filtered_color_img, cv2.COLOR_BGR2GRAY)

# Display the resized and filtered images using matplotlib
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(resized_img, cmap='gray')
plt.title("Resized Image")

plt.subplot(1, 2, 2)
plt.imshow(pyrMeanShiftFiltering_filtered_img, cmap='gray')
plt.title("Filtered Image")

plt.show()


# Scharr()

In [ ]:
# Assuming you have loaded or obtained the 'gray_img' properly
resized_img = cv2.resize(gray_img, (200, 200))

# Apply the Scharr filter in x and y directions
scharr_x = cv2.Scharr(resized_img, cv2.CV_16S, 1, 0)
scharr_y = cv2.Scharr(resized_img, cv2.CV_16S, 0, 1)

# Convert the filtered images back to CV_8U type
scharr_x_abs = cv2.convertScaleAbs(scharr_x)
scharr_y_abs = cv2.convertScaleAbs(scharr_y)

# Combine the x and y filtered images to get the final edge image
Scharr_filtered_img = cv2.addWeighted(scharr_x_abs, 0.5, scharr_y_abs, 0.5, 0)

# Display the resized and filtered images using matplotlib
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(resized_img, cmap='gray')
plt.title("Resized Image")

plt.subplot(1, 2, 2)
plt.imshow(Scharr_filtered_img, cmap='gray')
plt.title("Filtered Image (Scharr)")

plt.show()


# sqrBoxFilter()


In [ ]:
# Assuming you have loaded or obtained the 'gray_img' properly
resized_img = cv2.resize(gray_img, (200, 200))

ksize = 5
sigma = 3
theta = 1 * np.pi/4
lamda = 1 * np.pi/4  # Wavelength
gamma = 0.5
phi = 0  # phase offset

# Create a kernel using cv2.sqrBoxFilter()
kernel_size = (ksize, ksize)
kernel = cv2.sqrBoxFilter(np.ones(kernel_size, dtype=np.float32), -1, kernel_size)

# Apply the kernel using filter2D
sqrBoxFilter_filtered_img = cv2.filter2D(resized_img, cv2.CV_8U, kernel)

# Display the resized and filtered images using matplotlib
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(resized_img, cmap='gray')
plt.title("Resized Image")

plt.subplot(1, 2, 2)
plt.imshow(sqrBoxFilter_filtered_img, cmap='gray')
plt.title("Filtered Image")

plt.show()


# skimage.filters.threshold_li()

In [ ]:
import skimage.filters

# Assuming you have loaded or obtained the 'gray_img' properly
resized_img = cv2.resize(gray_img, (200, 200))

# Convert the image to a float type in the range [0, 1]
normalized_img = resized_img.astype(np.float32) / 255.0

# Apply Li's thresholding
threshold_value = skimage.filters.threshold_li(normalized_img)

# Create a binary image using the computed threshold
binary_img = normalized_img > threshold_value

# Display the original and thresholded images using matplotlib
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(normalized_img, cmap='gray')
plt.title("Original Image")

plt.subplot(1, 2, 2)
plt.imshow(binary_img, cmap='gray')
plt.title("Thresholded Image")

plt.show()


# skimage.filters.threshold_mean(image)

In [ ]:
from skimage import io, color, filters

resized_img = cv2.resize(gray_img, (200, 200))

# Apply mean thresholding
threshold_value = filters.threshold_mean(resized_img)

# Create a binary image based on the threshold value
binary_image = resized_img > threshold_value

# Display the original and thresholded images
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(resized_img, cmap='gray')
plt.title("Grayscale Image")

plt.subplot(1, 2, 2)
plt.imshow(binary_image, cmap='gray')
plt.title("Binary Image")

plt.show()


In [ ]:
# def create_gaborFilter():
#   filters = []
#   for theta in range(2):
#     theta = theta / 4.0 * np.pi
#     for sigma in (3, 5):
#       for lamda in np.arange(0, np.pi, np.pi/4):
#         for gamma in (0.05, 0.5):
#           #gaborLabel = 'Gabor' + str(count)
#           kernel = cv2.getGaborKernel((5, 5), sigma, theta, lamda, gamma, 0, ktype = cv2.CV_32F)
#           filters.append(kernel)
#           #filtered_IMG = cv2.filter2D(img, cv2.CV_8UC3, kernel)
#           #filtered_img = filtered_IMG.reshape(-1)
#           #df[gaborLabel] = filtered_img
#   return filters

In [ ]:
# def apply_filter(img, filters):
#   # This general function is designed to apply filters to our image
#   # First create a numpy array the same size as our input image

#   newimage = np.zeros_like(img)
#   # Starting with a blank image, we loop through the images and apply our Gabor Filter
#   # On each iteration, we take the highest value (super impose), until we have the max value across all filters
#   # The final image is returned
#   depth = -1 # remain depth same as original image

#   for kern in filters:  # Loop through the kernels in our GaborFilter
#     image_filter = cv2.filter2D(img, depth, kern)  #Apply filter to image

#     # Using Numpy.maximum to compare our filter and cumulative image, taking the higher value (max)
#     np.maximum(newimage, image_filter, newimage)   #(arr1, arr2, result)
#   return newimage

In [ ]:
# gfilters = create_gaborFilter()
# image_g = apply_filter(resized_img, gfilters)

# plt.imshow(resized_img,cmap='gray')
# plt.title("Resized Image")
# plt.show()

# plt.imshow(image_g, cmap='gray')
# plt.title("Filtered Image")
# plt.show()

In [ ]:
#trainingData['IDFeature'] = ""
#testingData['IDFeature'] = ""

training_ids = set(trainingData['Id'])
testing_ids = set(testingData['Id'])

for image in os.listdir("images/images"):
    image_id = int(image.split('.')[0])
    path = "images/images/" + image
    img = cv2.imread(path)
    image1 = cv2.resize(img, (100, 100))

    #Applying filters to all images:
    # gfilters = create_gaborFilter()
    # image_g = apply_filter(image1, gfilters)
    #***************************************************************
    # # Apply bilateral filter
    # d = 9         # Diameter of pixel neighborhood
    # sigmaColor = 75  # Filter sigma in the color space
    # sigmaSpace = 75  # Filter sigma in the coordinate space
    # bilateralFilter_filtered_img = cv2.bilateralFilter(image1, d, sigmaColor, sigmaSpace)
    #***************************************************************
    ksize = 5
    # Apply median blur using cv2.medianBlur
    medianBlur_filtered_img = cv2.medianBlur(image1, ksize)
    
    
    # gray_img = cv2.cvtColor(bilateralFilter_filtered_img, cv2.COLOR_BGR2GRAY)
    img2 = medianBlur_filtered_img.reshape(1, -1)

  # image_id = int(image.split('.')[0])
  # path = "images/" + image
  # img = io.imread(path, as_gray = True)
  # img2 = img.reshape(1, -1) #Row

    if image_id in training_ids:
        for train_id in training_ids:
            if image_id == train_id:
                row_index = trainingData.index[trainingData['Id'] == image_id][0]
                # trainingData.at[row_index, 'IDFeature'] = image_id
                # Save the pixel values as separate features
                for idx, pixel_value in enumerate(img2.flatten()):
                    feature_name = f"img_feature{idx}"
                    trainingData.at[row_index, feature_name] = pixel_value
                break


    if image_id in testing_ids:
        for test_id in testing_ids:
            if image_id == test_id:
                row_index = testingData.index[testingData['Id'] == image_id][0]
                #testingData.at[row_index, 'IDFeature'] = image_id
                for idx, pixel_value in enumerate(img2.flatten()):
                    feature_name = f"img_feature{idx}"
                    testingData.at[row_index, feature_name] = pixel_value
                break

trainingData.to_csv("medianBlur_train.csv", index = False)
testingData.to_csv("medianBlur_test.csv", index = False)

In [ ]:
#Saving Normal Images in NormalImagesFolder & BenignImages:
training_ids = set(trainingData['Id'])
testing_ids = set(testingData['Id'])

for image in os.listdir("images/images"):
    image_id = int(image.split('.')[0])
    path = "images/images/" + image
    img = cv2.imread(path)
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    if image_id in training_ids:
        for train_id in training_ids:
            if image_id == train_id:
                row_index = trainingData.index[trainingData['Id'] == image_id][0]
                class_label = trainingData.loc[row_index, 'Class']
                if class_label == "Malignant":
                    image_filename = os.path.basename(path)
                    target_image_path = os.path.join("MalignantImages", image_filename)
                    shutil.copy2(path, target_image_path)
                    break

In [ ]:
training_ids = set(trainingData['Id'])
testing_ids = set(testingData['Id'])

for image in os.listdir("images/images"):
    image_id = int(image.split('.')[0])
    path = "images/images/" + image
    img = cv2.imread(path)
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    if image_id in training_ids:
        for train_id in training_ids:
            if image_id == train_id:
                row_index = trainingData.index[trainingData['Id'] == image_id][0]
                class_label = trainingData.loc[row_index, 'Class']
                if class_label == "Malignant":
                    image_filename = os.path.basename(path)
                    target_image_path = os.path.join("MalignantImages", image_filename)
                    shutil.copy2(path, target_image_path)
                    break


In [ ]:
#trainingData['Class'] = trainingData['Class'].map({0 : "Normal", 1 : "Benign", 2 : "Malignant"})
trainingData.shape

In [ ]:
trainingDataOriginal = pd.read_csv("bilateralFilter_train.csv")
trainingDataOriginal.head()

In [ ]:
trainingDataGabor = pd.read_csv("updated_training_GaborFilter.csv")
trainingDataGabor.head()

In [ ]:
trainingDataOriginal.shape

In [ ]:
x = trainingDataOriginal.drop(columns = ['Unnamed: 0', 'Class', 'Id', 'IDFeature'])
y = trainingDataOriginal['Class']

X_train, X_val, y_train, y_val = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
x.shape

In [ ]:
pca = PCA(n_components = 80)

X_train = pca.fit_transform(X_train)
X_val = pca.transform(X_val)

In [ ]:
x.shape

In [ ]:
X_train.shape

In [ ]:
#Random Forest
rf = RandomForestClassifier(random_state = 42)

rf.fit(X_train, y_train)
predictions = rf.predict(X_val)
print(classification_report(y_val, predictions))

In [ ]:
test = pd.read_csv("updated_testingOriginalP.csv")

testData = test.drop(columns = ['Unnamed: 0', 'Id', 'IDFeature'])
#predictedV = rf.predict(testData)

In [ ]:
testData.shape

In [ ]:
testData = pca.transform(testData)
predictedV = rf.predict(testData)

In [ ]:
IDDF = testingData['Id']

predictionsDF = pd.DataFrame({'Id': IDDF, 'Class': predictedV})

#trainingData.to_csv("updated_trainingOriginalP.csv", index = False)
predictionsDF.to_csv("submission#3.csv", index=False)

In [ ]:
#LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_val)
print(classification_report(y_val, predictions))

In [ ]:
#!pip install Augmentor

In [ ]:
# p = Augmentor.Pipeline("image1")

# p.rotate(probability=0.7, max_left_rotation=10, max_right_rotation=10)
# p.zoom(probability=0.5, min_factor=1.1, max_factor=1.5)
# p.sample(10000)

In [ ]:
img = cv2.imread("images/1006.png")

img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# get the image shape
rows, cols = img.shape

#transformation matrix for reflection
M = np.float32([[-1, 0, cols],
                [ 0, 1, 0   ],
                [ 0, 0, 1   ]])

# apply a perspective transformation to the image
translated_img = cv2.warpPerspective(img, M, (cols, rows))

#cv2_imshow(img)
print(img.shape)
# show the resulting image
#cv2_imshow(translated_img)
print(translated_img.shape)